In [10]:
# IMPORTS

import argparse
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.callbacks import EarlyStopping

In [11]:



# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
np.random.seed(SEED)

FEATURE_COLUMNS = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
]

TARGET_COLUMN = "koi_disposition"


# ============================================================
# DATA DOWNLOAD
# ============================================================

def download_koi_data(output_path="data/koi_exoplanet_data.csv"):
    Path("data").mkdir(parents=True, exist_ok=True)

    selected_columns = ", ".join([TARGET_COLUMN] + FEATURE_COLUMNS)

    query = f"""
    SELECT {selected_columns}
    FROM cumulative
    WHERE koi_disposition IS NOT NULL
    """

    encoded_query = quote(query)

    url = (
        "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
        f"?query={encoded_query}"
        "&format=csv"
    )

    print("Downloading NASA Exoplanet Archive KOI data...")

    response = requests.get(url, timeout=60)

    if response.status_code != 200:
        raise RuntimeError(
            f"Data download failed with status code {response.status_code}"
        )

    Path(output_path).write_text(response.text, encoding="utf-8")

    df = pd.read_csv(output_path)

    if df.empty:
        raise ValueError("Downloaded dataset is empty.")

    return df


# ============================================================
# PREPROCESSING
# ============================================================

def preprocess_data(df):
    df = df.copy()
    df = df[[TARGET_COLUMN] + FEATURE_COLUMNS]
    df = df.dropna()

    X = df[FEATURE_COLUMNS].values
    y = df[TARGET_COLUMN].values

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, y_encoded, label_encoder, scaler


# ============================================================
# MODEL DEFINITIONS
# ============================================================

def build_mlp_model(input_dim, num_classes):
    model = Sequential([
        Dense(256, activation="relu", input_shape=(input_dim,)),
        Dropout(0.35),

        Dense(128, activation="relu"),
        Dropout(0.30),

        Dense(64, activation="relu"),
        Dropout(0.20),

        Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


def build_cnn_model(input_shape, num_classes):
    model = Sequential([
        Conv1D(64, kernel_size=3, activation="relu", input_shape=input_shape),
        MaxPooling1D(pool_size=2),

        Conv1D(128, kernel_size=3, activation="relu"),
        Flatten(),

        Dense(128, activation="relu"),
        Dropout(0.35),

        Dense(64, activation="relu"),
        Dropout(0.20),

        Dense(num_classes, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


def prepare_model(model_type, X_train, X_test, num_classes):
    if model_type == "mlp":
        model = build_mlp_model(X_train.shape[1], num_classes)
        return model, X_train, X_test

    if model_type == "cnn":
        X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
        X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
        model = build_cnn_model((X_train.shape[1], 1), num_classes)
        return model, X_train_cnn, X_test_cnn

    raise ValueError("model_type must be either mlp or cnn")


# ============================================================
# VISUALIZATION
# ============================================================

def plot_training_history(history, output_path="results/plots/training_history.png"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    plt.figure()
    plt.plot(history.history["accuracy"], label="Training Accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Metric Value")
    plt.title("Training and Validation Performance")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()


def plot_confusion_matrix(
    y_true,
    y_pred,
    class_names,
    output_path="results/plots/confusion_matrix.png",
):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    plt.imshow(cm)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(np.arange(len(class_names)), class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar()
    plt.tight_layout()
    plt.savefig(output_path)
    plt.close()


# ============================================================
# TRAINING PIPELINE
# ============================================================

def run_pipeline(args):
    df = download_koi_data()

    X, y, label_encoder, scaler = preprocess_data(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=SEED,
        stratify=y,
    )

    num_classes = len(np.unique(y))

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train,
    )

    class_weight_dict = {
        i: weight for i, weight in enumerate(class_weights)
    }

    model, X_train_model, X_test_model = prepare_model(
        args.model,
        X_train,
        X_test,
        num_classes,
    )

    early_stop = EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        restore_best_weights=True,
        mode="max",
    )

    history = model.fit(
        X_train_model,
        y_train,
        validation_data=(X_test_model, y_test),
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=[early_stop],
        class_weight=class_weight_dict,
        verbose=1,
    )

    probabilities = model.predict(X_test_model)
    y_pred = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_test, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    print("\nEvaluation Metrics")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")

    print("\nClassification Report")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=label_encoder.classes_,
            zero_division=0,
        )
    )

    Path("results/models").mkdir(parents=True, exist_ok=True)

    model_path = f"results/models/exoplanet_{args.model}_model.keras"
    model.save(model_path)

    plot_training_history(history)
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_)

    print(f"\nModel saved to {model_path}")
    print("Plots saved to results/plots/")


# ============================================================
# ARGUMENT PARSER
# ============================================================

def parse_args(args_list=None):
    parser = argparse.ArgumentParser(
        description="Deep learning exoplanet classification using NASA KOI data"
    )

    parser.add_argument(
        "--model",
        type=str,
        default="mlp",
        choices=["mlp", "cnn"],
    )

    parser.add_argument(
        "--epochs",
        type=int,
        default=100,
    )

    parser.add_argument(
        "--batch_size",
        type=int,
        default=32,
    )

    return parser.parse_args(args_list)


# ============================================================
# RUN PROJECT
# ============================================================

args = parse_args([])
run_pipeline(args)

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


230/230 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.5883 - loss: 0.9176 - val_accuracy: 0.6361 - val_loss: 0.7973
Epoch 2/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6387 - loss: 0.8233 - val_accuracy: 0.6486 - val_loss: 0.7820
Epoch 3/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6556 - loss: 0.7974 - val_accuracy: 0.6649 - val_loss: 0.7270
Epoch 4/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6630 - loss: 0.7814 - val_accuracy: 0.6697 - val_loss: 0.7284
Epoch 5/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6682 - loss: 0.7742 - val_accuracy: 0.6860 - val_loss: 0.7135
Epoch 6/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6727 - loss: 0.7633 - val_accuracy: 0.6697 - val_loss: 0.7064
Epoch 7/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6768 - loss: 0.7580 - val_accuracy: 0.6757 - val_loss: 0.6945
Epoch 8/100
230/230 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6773 - loss: 0.7516 - val_accuracy: 0.6757